## Function/Tool Calling for Airline Booking

In [375]:
import ollama
# import json
import pprint


In [376]:
# Function for getting the prices of a Flight
MODEL = "llama3.1:8B"

import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
                    CREATE TABLE IF NOT EXISTS 
                    prices (city TEXT PRIMARY KEY,
                            price REAL
                            )
    """)



def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('select price from prices where city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
    
        return f"The price of a ticket to {destination_city} is ${result}" if result else "No price available for this city."
    
    # price = ticket_prices.get(destination_city.lower(), "Unknown Ticket Price")
    # return f"The price of a ticket to {destination_city} is {price}"

In [377]:
# def set_ticket_price(destination_city, price):
#     print(f"Tool called for setting ticket Price for {destination_city} as {price}")
#     if(ticket_prices.get(destination_city.lower(),"Unknown Ticket Price")):
#         pass
#     else:
#         ticket_prices[destination_city] = "$"+price
#         content = "The ticket price has been set as desired...YOOOhOOOOOO\n\n"
#         print(content)
#         return content
def set_ticket_price(destination_city, price):
    print(f"Tool called for setting ticket price for {destination_city} as {price}")

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("insert into prices (city,price) values (?,?) on conflict(city) do update set price = ?", (destination_city.lower(), price, price))
        conn.commit()
    # ticket_prices[destination_city.lower()] = "$" + str(price)
    content = f"The ticket price for {destination_city} has been set to ${price}."
    # print(content)
    return content


In [378]:
# Below is th dictionary structure needed to describe our function in order for the llm to match the user query to our function

# This is a Function/TOOL Definition Schema.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer want to travel to"
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a ticket to the destination city when the name of destination city and the price is given as input",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer want to set the price for"
            },
            "price": {
                "type": "int",
                "description": "Price to be set for the destination city"
            }
        },
        "required": ["destination_city","price"],
        "additionalProperties": False
    }
}

In [379]:
# During the llm call we need to pass the messages with the concatenated history and also another
# essential field called the tools
# this is a essential feature that modern frontier models support even the open source one.

#You can see below we pass the function description and not the actual function
# tools = [get_ticket_price]

# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function},{"type":"function","function":set_price_function}]

In [380]:
# system_prompt = """
# You are a concise and courteous assistant for FlightAI. You can converse with the client as a normal human assistant and You support only two functions as given below:

# 1. Get the price of a ticket to a destination city. This requires only the city name.
# 2. Set the ticket price for a destination city. This requires the city name and the price.

# You must:
# - Use the tools listed in the tools=tools field.
# - Call a tool immediately once all required parameters are available.
# - Do not ask for travel dates, cabin class, or any other information not defined in the tool schema.
# - If the user asks anything outside these two functions, respond with: "I will not be able to help in that regard.

# In the end if queries have completed and the finish reason is not tool_call then reply with success message for the queries mention in the most current response returned from the tool

# """
system_prompt = """
You are a concise and courteous assistant for FlightAI. You can converse naturally with the user.

You support ONLY the following two functions:
1. Get the price of a ticket to a destination city (requires: city name).
2. Set the ticket price for a destination city (requires: city name and price).

Rules you must follow:
- Use only the tools provided in the tools=tools field.
- Call a tool immediately once all required parameters are available.
- Do NOT ask for extra information such as travel dates, class, etc.
- If the user asks anything outside these two functions, respond exactly with:
  "I will not be able to help in that regard."
- After completing all tool calls, if the finish_reason is NOT "tool_calls", return a success message summarizing the results from the most recent tool responses.

-----------------------------------
EXAMPLES:

Example 1:
User: What is the price of a ticket to Paris?
Assistant: (calls tool: get_ticket_price with city="paris")
Tool Response: 500
Assistant: Success: The ticket price for Paris is 500.

-----------------------------------

Example 2:
User: Set the ticket price for Tokyo to 800.
Assistant: (calls tool: set_ticket_price with city="tokyo", price=800)
Tool Response: OK
Assistant: Success: The ticket price for Tokyo has been set to 800.

-----------------------------------

Example 3:
User: What is the price of a ticket to Delhi and Mumbai?
Assistant:
(calls tool: get_ticket_price with city="delhi")
(calls tool: get_ticket_price with city="mumbai")

Tool Responses:
Delhi → 300
Mumbai → 400

Assistant:
Success: The ticket price for Delhi is 300 and for Mumbai is 400.

-----------------------------------

Example 4:
User: Set the price for Bangalore to 600 and Chennai to 550, and tell me the price for Hyderabad.
Assistant:
(calls tool: set_ticket_price with city="bangalore", price=600)
(calls tool: set_ticket_price with city="chennai", price=550)
(calls tool: get_ticket_price with city="hyderabad")

Tool Responses:
Bangalore → OK
Chennai → OK
Hyderabad → 450

Assistant:
Success: The ticket price for Bangalore has been set to 600, for Chennai has been set to 550, and the ticket price for Hyderabad is 450.

-----------------------------------

Example 5:
User: Book me a flight to London tomorrow.
Assistant:
I will not be able to help in that regard.

-----------------------------------

Output Style Rules:
- Always start final responses with "Success:"
- Combine multiple results into a single sentence.
- Be concise and clear.
"""

In [381]:
# lets populate the database
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

for city,price in ticket_prices.items():
    set_ticket_price(city,price)


Tool called for setting ticket price for london as $799
Tool called for setting ticket price for paris as $899
Tool called for setting ticket price for tokyo as $1400
Tool called for setting ticket price for berlin as $499


### USING OLLAMA CLIENT

In [382]:
# def handle_tool_calls(message):
#     responses = []
#     pprint.pprint(message)
#     for tool_call in message.tool_calls:
#         if tool_call.function.name == "get_ticket_price":
#             arguments = tool_call.function.arguments
#             city = arguments.get("destination_city")
#             price_details = get_ticket_price(city)
#             responses.append({
#                 "role":"tool",
#                 "content": price_details
#             })
#         if tool_call.function.name == "set_ticket_price":                        
#             arguments = tool_call.function.arguments
#             city1 = arguments.get('destination_city')
#             price = arguments.get('price')
#             content = set_ticket_price(city1,price)
#             # print(content)
#             responses.append({
#                 "role":"tool",
#                 "content":content,
#                 # "tool_call_id":tool_call.id
#             })
#     return responses

In [383]:
# def chat(message,history):
#     print(message,"\n")
#     print("____________________________________________________________\n_________________________________________")
#     print(history)
#     history = [{"role":h["role"],"content":h["content"]} for h in history]
#     messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user","content":message}]
#     response = ollama.chat(model=MODEL,messages=messages,tools=tools)

#     while response.message.tool_calls:
#         message = response.message
#         responses = handle_tool_calls(message)
#         messages.append({
#             "role": "assistant",
#             "content": message.content
#         })
#         messages.extend(responses)
#         response = ollama.chat(model=MODEL, messages = messages)
#     return response.message.content

In [384]:
# import gradio as gr

# gr.ChatInterface(fn=chat).launch()

### USING OPENAI CLIENT

In [385]:
from openai import OpenAI
import json

In [386]:
MODEL = "llama3.1:8b"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [387]:
def handle_tool_calls1(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":                        
            arguments = json.loads(tool_call.function.arguments)
            city1 = arguments.get('destination_city')
            price = arguments.get('price')
            content = set_ticket_price(city1,price)
            print(content)
            responses.append({
                "role":"tool",
                "content":content,
                "tool_call_id":tool_call.id
            })
    print(responses)

    return responses

In [ ]:
# chat function for connecting ollama using openAI client

# THe history is being forgotten for every gradio callback to the chat(message,history) function. It works fine for this experiment but for commercial purposes
# a database should be used to store the conversation history.


def chat(message, history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model=MODEL,messages = messages, tools = tools)
    print(response.choices[0])
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message # this gets printed first
        responses = handle_tool_calls1(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL,messages=messages, tools=tools)
        print("____________________",response.choices[0])
    
    return response.choices[0].message.content


    # When there is a tool call no content will be generated for that response generated by LLM. Instead the finish/stopping reason of llm will be TOOL CALLED.


In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7914
* To create a public link, set `share=True` in `launch()`.


Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_y5104ubw', function=Function(arguments='{"destination_city":"zimbabwe"}', name='get_ticket_price'), type='function', index=0)]))
Tool called for city zimbabwe
[{'role': 'tool', 'content': 'The price of a ticket to zimbabwe is $(140.0,)', 'tool_call_id': 'call_y5104ubw'}]
____________________ Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Success: The ticket price for Zimbabwe is 140.0.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))


In [ ]:
##future exercise idea.
#  Create a booing.py and use  it to book ticket using tool calling